In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"), temperature=0.4)

c:\Users\harih\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun(description="Search the web for current news, trends, and general information.")

In [5]:
print(search_tool.invoke("latest AI news today"))

AI news delivered 3x/week. Curated AI updates on funding, models, regulation, and applications. Read by 50,000+ professionals since 2015. Subscribe free. Get the latest AI news today from ainewstodays.com. Stay updated on artificial intelligence, machine learning, tech, business, and innovation trends.Your daily source for the latest AI breakthroughs, product launches, and industry shifts — delivered fast, verified, and straight to the point. This week's AI news roundup covers Lovable's Google Cloud deal, Railway's $100M raise, Walmart's AI cost realities, and E.ON's grid modernisation with SAP AI. 4 Jun 2026 4 min read. Read full story. Today's top stories. Behind the scenes at The Economist. Newsletters. Curated news, direct to your inbox. Games. Workouts for agile minds.Britain. Health tech and AI come to equestrianism. A heritage sport is enjoying a data revolution, with animal welfare as the justification. The latest AI news from June.Get the latest news from Google in your inbox.

In [6]:
import wikipedia
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

wikipedia.set_user_agent("SecondBrainApp/1.0 (contact: training@example.com)")

wiki_api = WikipediaAPIWrapper()
wikipedia_tool = WikipediaQueryRun(api_wrapper=wiki_api)

In [7]:
print(wikipedia_tool.invoke("What is Retrieval-Augmented Generation?"))

Page: Retrieval-augmented generation
Summary: Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this enables LLM-based chatbots to access internal company data or generate responses based on authoritative sources. The technique was first proposed in 2020 and has since become a widely adopted approach in modern AI systems.
RAG improves LLMs by incorporating information retrieval before generating responses. Unlike LLMs that rely on static training data, RAG pulls relevant text from databases, uploaded documents, or web sources. According to Ars Technica, "RAG is a way of improving L

In [8]:
from langchain.agents import create_agent

research_toolkit = [search_tool, wikipedia_tool]

agent = create_agent(llm, tools=research_toolkit)

In [9]:
from langchain.agents import create_agent

research_toolkit = [search_tool, wikipedia_tool]

agent = create_agent(llm, tools=research_toolkit)

In [12]:
from langchain.agents import create_agent

research_toolkit = [search_tool, wikipedia_tool]

system_prompt = """You are a research assistant with two tools:
- duckduckgo_search: for current events, news, and anything time-sensitive.
- wikipedia_query_run: for background, definitions, and established concepts.

Rules:
- Use each tool AT MOST ONCE per request unless the first result is clearly empty or broken.
- Do not repeat a search with slightly reworded queries — if the first result has useful content, use it.
- Match the tool to the type of question asked.
- After gathering what you need, answer directly. Do not keep calling tools "to be safe"."""

agent = create_agent(llm, tools=research_toolkit, prompt=system_prompt)

TypeError: create_agent() got an unexpected keyword argument 'prompt'

In [13]:
help(create_agent)

Help on function create_agent in module langchain.agents.factory:

create_agent(
    model: str | BaseChatModel,
    tools: Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None = None,
    *,
    system_prompt: str | SystemMessage | None = None,
    middleware: Sequence[AgentMiddleware[StateT_co, ContextT]] = (),
    response_format: ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None = None,
    state_schema: type[AgentState[ResponseT]] | None = None,
    context_schema: type[ContextT] | None = None,
    checkpointer: Checkpointer | None = None,
    store: BaseStore | None = None,
    interrupt_before: list[str] | None = None,
    interrupt_after: list[str] | None = None,
    debug: bool = False,
    name: str | None = None,
    cache: BaseCache[Any] | None = None,
    transformers: Sequence[TransformerFactory] | None = None
) -> CompiledStateGraph[AgentState[ResponseT], ContextT, InputAgentState, OutputAgentState[ResponseT]]
    Creates an agent graph that 

In [14]:
from langchain.agents import create_agent

research_toolkit = [search_tool, wikipedia_tool]

system_prompt = """You are a research assistant with two tools:
- duckduckgo_search: for current events, news, and anything time-sensitive.
- wikipedia_query_run: for background, definitions, and established concepts.

Rules:
- Use each tool AT MOST ONCE per request unless the first result is clearly empty or broken.
- Do not repeat a search with slightly reworded queries — if the first result has useful content, use it.
- Match the tool to the type of question asked.
- After gathering what you need, answer directly. Do not keep calling tools "to be safe"."""

agent = create_agent(llm, tools=research_toolkit, system_prompt=system_prompt)

In [15]:
example_query = "1) What is one current AI news headline from this week? 2) Separately, explain what a transformer model is in AI."

events = agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

1) What is one current AI news headline from this week? 2) Separately, explain what a transformer model is in AI.
================================== Ai Message ==================================
Tool Calls:
  duckduckgo_search (fc_7391692d-8133-4b26-80c8-aa1c0426f381)
 Call ID: fc_7391692d-8133-4b26-80c8-aa1c0426f381
  Args:
    query: AI news headline this week 2024 August
================================= Tool Message =================================
Name: duckduckgo_search

Read today’s latest world news for all the breaking international stories from Europe, Asia, the Middle East, and more, on the New York Post.The health-threatening AI tech banned in EU this week — while US remains unregulated. In 2024 US week numbers match ISO week numbers for most dates (except on Sundays and around the turn of the year). Find more info on our main week numbers page.August 11, 2024. Week 33. Explore today's Britis

KeyboardInterrupt: 

In [16]:
from datetime import date

today_str = date.today().strftime("%B %d, %Y")

system_prompt = f"""You are a research assistant. Today's date is {today_str}.

You have two tools:
- duckduckgo_search: for current events, news, and anything time-sensitive.
- wikipedia_query_run: for background, definitions, and established concepts.

STRICT RULES:
- Call duckduckgo_search AT MOST ONE TIME total, no matter what it returns.
- Call wikipedia_query_run AT MOST ONE TIME total.
- Never repeat a tool call with a reworded query.
- After your tool calls, answer with whatever information you have, even if imperfect.
- Do not say a search "failed" and retry — just use the best result you got."""

agent = create_agent(llm, tools=research_toolkit, system_prompt=system_prompt)

In [17]:
example_query = "1) What is one current AI news headline from this week? 2) Separately, explain what a transformer model is in AI."

events = agent.stream(
    {"messages": [("user", example_query)]},
    config={"recursion_limit": 8},
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

1) What is one current AI news headline from this week? 2) Separately, explain what a transformer model is in AI.
================================== Ai Message ==================================
Tool Calls:
  duckduckgo_search (fc_81a7740d-0a47-4551-a3b6-30c3ee66dd0e)
 Call ID: fc_81a7740d-0a47-4551-a3b6-30c3ee66dd0e
  Args:
    query: AI news headline this week August 2026
================================= Tool Message =================================
Name: duckduckgo_search

Here's the news from ANC Headlines this August 7, 2026. The latest AI news we announced in July 2026.To keep you posted on our progress, we're doing a regular roundup of Google's most recent AI news. Here’s a look back at some of our AI announcements from July. Read today’s latest world news for all the breaking international stories from Europe, Asia, the Middle East, and more, on the New York Post.The health-threatening AI tech b

In [18]:
research_toolkit = [search_tool, wikipedia_tool]